# PlantDoc — ConvNeXt-Tiny (pré-entraîné ImageNet)

Ce notebook **complet** entraîne et évalue un classifieur ConvNeXt-Tiny
sur **PlantDoc** (images terrain) en partant des poids **ImageNet** :

1. ✅ Installer les dépendances
2. ✅ Charger PlantDoc depuis Hugging Face
3. ✅ Mapper et nettoyer les classes
4. ✅ Filtrer, vérifier la qualité, pré-redimensionner en RAM
5. ✅ Séparer (train/val/test) avec contrôle de fuite de données
6. ✅ Charger ConvNeXt-Tiny (poids ImageNet)
7. ✅ Entraîner (tête seule → fine-tuning complet)
8. ✅ Tester et évaluer (accuracy, macro-F1, confusion matrix, par plante)
9. ✅ Exporter le modèle

**Objectif** : évaluer les performances de ConvNeXt-Tiny
directement sur des images terrain (PlantDoc) sans passer par PlantVillage.


In [ ]:
%pip install -q "pandas==2.2.3" "pyarrow<24,>=19"
%pip install -q datasets huggingface_hub tqdm scikit-learn matplotlib seaborn opencv-python-headless onnx
# Do not upgrade torch or torchvision: keep Colab's CUDA-compatible versions.

## 1. Configuration et reproductibilité

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, random, time, json, shutil, warnings, zipfile
import numpy as np
import pandas as pd
import torch
from PIL import Image

# ── Reproductibilité ──
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)

DATASET_ID = 'geraldmc/plantdoc-full'
BACKBONE = 'convnext_tiny'

IMAGE_SIZE = 224
RESIZE_CACHE = 256          # pré-redimensionner en RAM à cette taille
BATCH_SIZE = 16
NUM_WORKERS = 2
MAX_EPOCHS = 20
HEAD_EPOCHS = 3
MAX_MINUTES_PER_SESSION = 90
CHECKPOINT_EVERY_STEPS = 100

WORK_ROOT = Path('/content/drive/MyDrive/plant-disease/plantdoc_classifier')
RUN_ROOT = WORK_ROOT / 'runs/convnext_tiny_imagenet'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('PlantDoc run directory:', RUN_ROOT)


## 2. Charger PlantDoc et mapper les classes

PlantDoc contient des images de terrain de 28 classes. On ne garde que les
classes qui correspondent aux classes PlantVillage de notre projet (14 classes
de pomme, maïs et tomate).

Le mapping est basé sur `configs/mappings/plantdoc.yaml` du projet.


In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_ID)

# Merge all splits into a single table with a 'split' column
# (geraldmc/plantdoc-full stores train/test as separate splits)
all_rows = []
for split_name in dataset.keys():
    ds_split = dataset[split_name]
    for idx in range(len(ds_split)):
        all_rows.append({'_hf_split': split_name, '_hf_idx': idx})

print(f'PlantDoc total: {len(all_rows)} rows across splits: {list(dataset.keys())}')

# ── Mapping PlantDoc → PlantVillage class names ──
PLANTDOC_TO_PLANTVILLAGE = {
    'Apple Scab Leaf':                      'Apple___Apple_scab',
    'Apple rust leaf':                       'Apple___Cedar_apple_rust',
    'Apple leaf':                            'Apple___healthy',
    'Corn Gray leaf spot':                   'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn rust leaf':                        'Corn_(maize)___Common_rust_',
    'Corn leaf blight':                      'Corn_(maize)___Northern_Leaf_Blight',
    'Tomato Early blight leaf':              'Tomato___Early_blight',
    'Tomato Septoria leaf spot':             'Tomato___Septoria_leaf_spot',
    'Tomato leaf bacterial spot':            'Tomato___Bacterial_spot',
    'Tomato leaf late blight':               'Tomato___Late_blight',
    'Tomato leaf yellow virus':              'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato leaf mosaic virus':              'Tomato___Tomato_mosaic_virus',
    'Tomato mold leaf':                      'Tomato___Leaf_Mold',
    'Tomato leaf':                           'Tomato___healthy',
    # Excluded: 'Tomato two spotted spider mites leaf' → only 2 images, not usable
}

# ── Host (plant) pour évaluation par plante ──
def host_from_class(cls_name):
    if cls_name.startswith('Apple'):     return 'Apple'
    if cls_name.startswith('Corn'):      return 'Corn'
    if cls_name.startswith('Tomato'):    return 'Tomato'
    return 'Other'

CLASS_NAMES = sorted(set(PLANTDOC_TO_PLANTVILLAGE.values()))
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
CLASS_HOSTS = [host_from_class(n) for n in CLASS_NAMES]

print(f'\nMapped to {NUM_CLASSES} classes:')
for idx, name in enumerate(CLASS_NAMES):
    print(f'  {idx}: {name}  [{CLASS_HOSTS[idx]}]')


## 3. Filtrer, vérifier la qualité, pré-redimensionner

**Optimisation** : on filtre d'abord sur les colonnes texte (classe, split)
sans décoder les images. Ensuite, on décode et pré-redimensionne
chaque image gardée une seule fois en mémoire (256×256), pour que le
DataLoader n'ait pas à redécoder des images de 4000 px à chaque époque.


In [ ]:
import collections, io
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# ── Step 1: Metadata-only filter (no image decoding) ──
print('Step 1: Filtering by class label (metadata only)...')
filtered_meta = []  # (hf_split, hf_idx, class_id, original_split)
excluded_classes = set()

for split_name in dataset.keys():
    ds_split = dataset[split_name]
    labels_col = ds_split['class_label']
    for idx, pd_label in enumerate(labels_col):
        if pd_label not in PLANTDOC_TO_PLANTVILLAGE:
            excluded_classes.add(pd_label)
            continue
        pv_name = PLANTDOC_TO_PLANTVILLAGE[pd_label]
        class_id = CLASS_TO_ID[pv_name]
        # Normalize split from HF split name
        orig_split = 'test' if 'test' in split_name.lower() else 'train'
        filtered_meta.append((split_name, idx, class_id, orig_split))

print(f'  Après filtrage par classe: {len(filtered_meta)} images')
print(f'  Classes exclues ({len(excluded_classes)}): {sorted(excluded_classes)}')

# ── Step 2: Decode, quality-check, pre-resize into RAM ──
print('\nStep 2: Decoding + quality check + pre-resize to', RESIZE_CACHE, '...')
image_cache = {}  # key: (hf_split, hf_idx) → PIL Image (RESIZE_CACHE × RESIZE_CACHE)
valid_rows = []   # (cache_key, class_id, original_split, filename)
bad_images = 0

for hf_split, hf_idx, class_id, orig_split in tqdm(filtered_meta, desc='Decoding'):
    try:
        row = dataset[hf_split][hf_idx]
        img = row['image']
        if img is None:
            bad_images += 1
            continue
        img_rgb = img.convert('RGB')
        w, h = img_rgb.size
        if w < 32 or h < 32:
            bad_images += 1
            continue
        # Pre-resize to RESIZE_CACHE to avoid re-decoding large images every epoch
        img_resized = img_rgb.resize((RESIZE_CACHE, RESIZE_CACHE), Image.BILINEAR)
        cache_key = (hf_split, hf_idx)
        image_cache[cache_key] = img_resized
        # Try to get a filename for leak detection
        fname = row.get('image_file_path', row.get('filename', ''))
        if isinstance(fname, str):
            fname = Path(fname).name  # just basename
        else:
            fname = ''
        valid_rows.append((cache_key, class_id, orig_split, fname))
    except Exception:
        bad_images += 1

print(f'\n✅ Valid images cached: {len(valid_rows)}')
print(f'❌ Bad/tiny images removed: {bad_images}')


## 4. Séparer en train / val / test (avec contrôle de fuite)

On utilise la colonne `split` du dataset (train/test).
On crée la validation = 15% du train, stratifiée.
On vérifie qu'aucun fichier n'est à la fois dans train et test.


In [ ]:
# ── Separate by original split ──
train_all = [(ck, cid, fn) for ck, cid, sp, fn in valid_rows if sp == 'train']
test_rows = [(ck, cid, fn) for ck, cid, sp, fn in valid_rows if sp == 'test']

print(f'Split du dataset:')
print(f'  train (original): {len(train_all)}')
print(f'  test  (original): {len(test_rows)}')

# ── Leak detection: check for common filenames ──
train_fnames = {fn for _, _, fn in train_all if fn}
test_fnames  = {fn for _, _, fn in test_rows if fn}
common_fnames = train_fnames & test_fnames
print(f'\n🔍 Noms de fichiers communs train↔test: {len(common_fnames)}')
if common_fnames:
    print(f'  ⚠️ Fuite de données détectée ! Retrait de {len(common_fnames)} images du train.')
    train_all = [(ck, cid, fn) for ck, cid, fn in train_all if fn not in common_fnames or not fn]
    print(f'  train après nettoyage: {len(train_all)}')
else:
    print('  ✅ Aucune fuite de données.')

# ── Fallback: if test is empty, create from train ──
if len(test_rows) == 0:
    print('\n⚠️ Pas de split test ! Création à partir du train (10%)...')
    train_labels = [cid for _, cid, _ in train_all]
    train_all, test_rows = train_test_split(
        train_all, test_size=0.10, random_state=SEED, stratify=train_labels)

# ── Create validation from train (15%, stratified) ──
train_labels = [cid for _, cid, _ in train_all]
train_rows, val_rows = train_test_split(
    train_all, test_size=0.15, random_state=SEED, shuffle=True, stratify=train_labels)

# Convert to simpler format: (cache_key, class_id)
split_rows = {
    'train': [(ck, cid) for ck, cid, _ in train_rows],
    'val':   [(ck, cid) for ck, cid, _ in val_rows],
    'test':  [(ck, cid) for ck, cid, _ in test_rows],
}

print(f'\n📊 Splits finaux:')
for split, rows in split_rows.items():
    counts = collections.Counter(cid for _, cid in rows)
    print(f'  {split}: {len(rows)} images, '
          f'per-class min/max: {min(counts.values()) if counts else 0}'
          f'/{max(counts.values()) if counts else 0}')

# ── Show test images per class ──
print(f'\n📋 Images de test par classe:')
test_counts = collections.Counter(cid for _, cid in split_rows['test'])
for idx, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {test_counts.get(idx, 0)}')

if any(len(split_rows[s]) == 0 for s in ('train', 'val', 'test')):
    raise RuntimeError('All splits must be non-empty')


## 5. Dataset, augmentation et DataLoaders

L'augmentation est plus agressive que pour PlantVillage (dataset plus petit).
Les images sont déjà pré-redimensionnées en mémoire → pas de décodage à chaque époque.

**Déséquilibre** : on utilise `CrossEntropyLoss(weight=...)` avec `shuffle=True`,
sans `WeightedRandomSampler`, pour ne pas compenser deux fois.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

class PlantDocDataset(Dataset):
    def __init__(self, cache, rows, training=False):
        self.cache = cache       # (hf_split, hf_idx) → PIL Image (RESIZE_CACHE)
        self.rows = rows         # [(cache_key, class_id), ...]
        self.training = training

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, position):
        cache_key, class_id = self.rows[position]
        image = self.cache[cache_key].copy()  # already RESIZE_CACHE×RESIZE_CACHE RGB
        image = TF.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)
        if self.training:
            if random.random() < 0.5:
                image = TF.hflip(image)
            if random.random() < 0.3:
                image = TF.vflip(image)
            if random.random() < 0.4:
                angle = random.uniform(-15, 15)
                image = TF.rotate(image, angle)
            if random.random() < 0.4:
                image = TF.adjust_brightness(image, random.uniform(0.7, 1.3))
            if random.random() < 0.4:
                image = TF.adjust_contrast(image, random.uniform(0.7, 1.3))
            if random.random() < 0.3:
                image = TF.adjust_saturation(image, random.uniform(0.7, 1.3))
        tensor = TF.to_tensor(image)
        tensor = TF.normalize(tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        return tensor, torch.tensor(class_id, dtype=torch.long)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# ── Class weights for CrossEntropyLoss (no sampler) ──
train_counts = np.bincount(
    [cid for _, cid in split_rows['train']], minlength=NUM_CLASSES
).astype(np.float64)
class_weights = train_counts.sum() / np.maximum(train_counts * NUM_CLASSES, 1.0)
print('Class weights:', {CLASS_NAMES[i]: f'{w:.2f}' for i, w in enumerate(class_weights)})

generator = torch.Generator().manual_seed(SEED)

train_dataset = PlantDocDataset(image_cache, split_rows['train'], training=True)
val_dataset   = PlantDocDataset(image_cache, split_rows['val'])
test_dataset  = PlantDocDataset(image_cache, split_rows['test'])

loaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=True,
                       persistent_workers=NUM_WORKERS > 0, worker_init_fn=seed_worker,
                       generator=generator, drop_last=True),
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True,
                     persistent_workers=NUM_WORKERS > 0),
    'test': DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=NUM_WORKERS > 0),
}
print({name: len(loader.dataset) for name, loader in loaders.items()})


## 6. Charger le modèle (poids ImageNet)

On charge ConvNeXt-Tiny avec les poids ImageNet par défaut, puis on
remplace la tête de classification (1000 → 14 classes).

**Stratégie en 2 phases :**
- Phase 1 (HEAD_EPOCHS) : backbone gelé, on entraîne seulement la nouvelle tête
- Phase 2 : tout le modèle est dégelé avec un learning rate plus bas pour le backbone


In [ ]:
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Build model from ImageNet weights ──
model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)
model.to(device)
print(f'✅ ConvNeXt-Tiny (ImageNet → {NUM_CLASSES} classes)')

criterion = nn.CrossEntropyLoss(weight=torch.from_numpy(class_weights).float().to(device))

def set_backbone_training(model, enabled):
    for parameter in model.features.parameters():
        parameter.requires_grad = enabled

set_backbone_training(model, False)

def build_optimizer(fine_tuning):
    if fine_tuning:
        set_backbone_training(model, True)
        return torch.optim.AdamW([
            {'params': model.features.parameters(), 'lr': 5e-5},
            {'params': model.classifier.parameters(), 'lr': 5e-4},
        ], weight_decay=1e-4)
    set_backbone_training(model, False)
    return torch.optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=1e-3, weight_decay=1e-4,
    )

optimizer = build_optimizer(False)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

def save_checkpoint(next_epoch, best_val_acc, history, global_step, name='latest.pt'):
    payload = {
        'next_epoch': next_epoch,
        'best_val_acc': best_val_acc,
        'history': history,
        'global_step': global_step,
        'backbone': BACKBONE,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'class_names': CLASS_NAMES,
        'config': {
            'image_size': IMAGE_SIZE,
            'normalization_mean': [0.485, 0.456, 0.406],
            'normalization_std': [0.229, 0.224, 0.225],
            'dataset': DATASET_ID,
            'num_classes': NUM_CLASSES,
        },
    }
    tmp = RUN_ROOT / f'.{name}.tmp'
    torch.save(payload, tmp)
    tmp.replace(RUN_ROOT / name)

# ── Resume from checkpoint if exists ──
history = []
best_val_acc = -1.0
global_step = 0
start_epoch = 0
latest = RUN_ROOT / 'latest.pt'
if latest.exists():
    state = torch.load(latest, map_location=device)
    if state.get('class_names') == CLASS_NAMES:
        model.load_state_dict(state['model'])
        history = state.get('history', [])
        best_val_acc = state.get('best_val_acc', -1.0)
        global_step = state.get('global_step', 0)
        start_epoch = state.get('next_epoch', 0)
        fine_tuning = start_epoch >= HEAD_EPOCHS
        optimizer = build_optimizer(fine_tuning)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(MAX_EPOCHS - start_epoch, 1))
        scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        print(f'Resuming epoch {start_epoch + 1}; best val accuracy={best_val_acc:.4f}')
    else:
        print('Checkpoint class list mismatch, starting fresh')
else:
    print(f'Starting fresh; backbone frozen for first {HEAD_EPOCHS} epochs')

print(device, f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params')


## 7. Entraînement

- Phase 1 : backbone gelé, on entraîne la tête
- Phase 2 : tout dégelé, fine-tuning complet
- Checkpoints après chaque époque + résumé automatique


In [ ]:
from tqdm.auto import tqdm

session_deadline = time.time() + MAX_MINUTES_PER_SESSION * 60
current_epoch = start_epoch
stage_initialized = start_epoch >= HEAD_EPOCHS

try:
    for epoch in range(start_epoch, MAX_EPOCHS):
        current_epoch = epoch
        if epoch == HEAD_EPOCHS and not stage_initialized:
            optimizer = build_optimizer(fine_tuning=True)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=max(MAX_EPOCHS - epoch, 1))
            stage_initialized = True

        model.train()
        running_loss = 0.0
        progress = tqdm(loaders['train'], desc=f'Epoch {epoch + 1}/{MAX_EPOCHS}')
        for images, labels in progress:
            if time.time() >= session_deadline:
                save_checkpoint(epoch, best_val_acc, history, global_step)
                print('Session time limit reached; resume later from latest.pt.')
                raise TimeoutError

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=device.type == 'cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            global_step += 1
            progress.set_postfix(loss=f'{loss.item():.4f}')
            if global_step % CHECKPOINT_EVERY_STEPS == 0:
                save_checkpoint(epoch, best_val_acc, history, global_step)

        # ── Validation ──
        model.eval()
        correct = total = 0
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in loaders['val']:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                with torch.autocast(device_type=device.type, dtype=torch.float16,
                                    enabled=device.type == 'cuda'):
                    logits = model(images)
                    val_loss += criterion(logits, labels).item()
                correct += (logits.argmax(1) == labels).sum().item()
                total += labels.numel()

        val_acc = correct / max(total, 1)
        row = {
            'epoch': epoch + 1,
            'train_loss': running_loss / max(len(loaders['train']), 1),
            'val_loss': val_loss / max(len(loaders['val']), 1),
            'val_accuracy': val_acc,
        }
        history.append(row)
        scheduler.step()

        save_checkpoint(epoch + 1, max(best_val_acc, val_acc), history, global_step)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint(epoch + 1, best_val_acc, history, global_step, 'best.pt')
        print(row)

except TimeoutError:
    pass
except KeyboardInterrupt:
    save_checkpoint(current_epoch, best_val_acc, history, global_step)
    raise

print(f'Completed through epoch {history[-1]["epoch"] if history else start_epoch}.')


In [ ]:
import matplotlib.pyplot as plt

if history:
    history_df = pd.DataFrame(history)
    display(history_df)
    history_df.to_csv(RUN_ROOT / 'history.csv', index=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    history_df.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', ax=ax1)
    ax1.set_title('Loss (train vs validation)'); ax1.grid()
    history_df.plot(x='epoch', y='val_accuracy', marker='o', ylim=(0, 1), ax=ax2)
    ax2.set_title('Validation Accuracy'); ax2.grid()
    plt.tight_layout(); plt.show()
else:
    print('No completed epoch yet. Rerun the training cell.')


## 8. Test et métriques

On évalue **une seule fois** le meilleur modèle (`best.pt`, choisi sur la validation)
sur le jeu de test PlantDoc (jamais vu pendant l'entraînement).

Métriques :
- Top-1 / Top-3 accuracy
- Classification report (precision, recall, **macro-F1**, support par classe)
- Confusion matrix
- **Précision par plante** (Apple, Corn, Tomato)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score, f1_score
import seaborn as sns

best = RUN_ROOT / 'best.pt'
if not best.exists():
    raise FileNotFoundError('No best.pt yet; complete at least one validation epoch')
state = torch.load(best, map_location=device)
model.load_state_dict(state['model'])
model.to(device)
model.eval()

all_labels = []
all_predictions = []
all_probabilities = []
with torch.no_grad():
    for images, labels in tqdm(loaders['test'], desc='test'):
        images = images.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=device.type == 'cuda'):
            logits = model(images)
        probs = logits.float().softmax(1).cpu().numpy()
        all_probabilities.append(probs)
        all_predictions.append(probs.argmax(1))
        all_labels.append(labels.numpy())

y_true = np.concatenate(all_labels)
y_pred = np.concatenate(all_predictions)
y_score = np.concatenate(all_probabilities)

top1 = (y_true == y_pred).mean()
top3 = top_k_accuracy_score(y_true, y_score, k=min(3, NUM_CLASSES),
                            labels=np.arange(NUM_CLASSES))
macro_f1 = f1_score(y_true, y_pred, labels=np.arange(NUM_CLASSES), average='macro',
                    zero_division=0)

print('=' * 70)
print('RÉSULTATS — ConvNeXt-Tiny (ImageNet) fine-tuné sur PlantDoc')
print('=' * 70)
print(f'\n🎯 Top-1 Accuracy: {top1:.4f} ({top1*100:.1f}%)')
print(f'🎯 Top-3 Accuracy: {top3:.4f} ({top3*100:.1f}%)')
print(f'🎯 Macro-F1 Score: {macro_f1:.4f}')
print(f'📊 Total test images: {len(y_true)}')
print()
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4,
                               labels=np.arange(NUM_CLASSES), zero_division=0)
print(report)

# ── Per-host (plant) accuracy ──
print('\n🌱 Précision par plante (host):')
host_metrics = {}
for host in ['Apple', 'Corn', 'Tomato']:
    host_class_ids = [i for i, h in enumerate(CLASS_HOSTS) if h == host]
    mask = np.isin(y_true, host_class_ids)
    if mask.sum() == 0:
        host_metrics[host] = {'accuracy': 0.0, 'support': 0}
        continue
    acc = (y_true[mask] == y_pred[mask]).mean()
    host_metrics[host] = {'accuracy': float(acc), 'support': int(mask.sum())}
    print(f'  {host:8s}: {acc:.4f} ({acc*100:.1f}%)  [{mask.sum()} images]')

# ── Confusion matrix ──
matrix = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
short_names = [n.replace('___', '\n').replace('_(maize)', '').replace('Corn\n', 'Corn\n')
               for n in CLASS_NAMES]
plt.figure(figsize=(14, 12))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.xlabel('Prédit'); plt.ylabel('Vrai')
plt.title('Matrice de confusion — ConvNeXt-Tiny (ImageNet) sur PlantDoc')
plt.tight_layout()
plt.savefig(RUN_ROOT / 'confusion_matrix.png', dpi=150)
plt.show()

# ── Support per class ──
support_per_class = {CLASS_NAMES[i]: int(test_counts.get(i, 0)) for i in range(NUM_CLASSES)}

# ── Save metrics ──
(RUN_ROOT / 'test_metrics.json').write_text(json.dumps({
    'top1_accuracy': float(top1),
    'top3_accuracy': float(top3),
    'macro_f1': float(macro_f1),
    'backbone': BACKBONE,
    'pretrained_on': 'imagenet',
    'dataset': DATASET_ID,
    'class_names': CLASS_NAMES,
    'support_per_class': support_per_class,
    'host_metrics': host_metrics,
    'classification_report': report,
    'seed': SEED,
}, indent=2) + '\n')
print('\n✅ Métriques sauvegardées dans', RUN_ROOT / 'test_metrics.json')


## 8b. (Optionnel) Multi-seed : moyenne et écart-type

Pour une évaluation plus robuste, on peut relancer l'entraînement avec
plusieurs seeds et afficher la moyenne ± écart-type de l'accuracy test.

**⚠️ Cette cellule est optionnelle et prend du temps (3× l'entraînement).**
Passer directement à la section 9 (Export) si un seul run suffit.


In [ ]:
# ⚠️ OPTIONNEL — décommenter et exécuter pour multi-seed
# Chaque seed ré-entraîne le modèle de zéro, cela prend ~3× plus de temps.

MULTI_SEEDS = [42, 123, 7]  # seeds à tester
RUN_MULTI_SEED = False      # Passer à True pour exécuter

if RUN_MULTI_SEED:
    from copy import deepcopy
    multi_results = []

    for s_idx, test_seed in enumerate(MULTI_SEEDS):
        print(f'\n{"="*50}')
        print(f'Multi-seed run {s_idx+1}/{len(MULTI_SEEDS)} — seed={test_seed}')
        print(f'{"="*50}')

        seed_everything(test_seed)

        # Rebuild train/val split (test stays the same)
        _train_labels = [cid for _, cid, _ in train_all]
        _train_rows, _val_rows = train_test_split(
            train_all, test_size=0.15, random_state=test_seed,
            shuffle=True, stratify=_train_labels)

        _split_rows = {
            'train': [(ck, cid) for ck, cid, _ in _train_rows],
            'val':   [(ck, cid) for ck, cid, _ in _val_rows],
            'test':  split_rows['test'],  # fixed
        }

        _gen = torch.Generator().manual_seed(test_seed)
        _train_ds = PlantDocDataset(image_cache, _split_rows['train'], training=True)
        _val_ds   = PlantDocDataset(image_cache, _split_rows['val'])
        _test_ds  = PlantDocDataset(image_cache, _split_rows['test'])

        _loaders = {
            'train': DataLoader(_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True,
                               worker_init_fn=seed_worker, generator=_gen, drop_last=True),
            'val': DataLoader(_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True),
            'test': DataLoader(_test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True),
        }

        # Fresh model
        _model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        _model.classifier[-1] = nn.Linear(_model.classifier[-1].in_features, NUM_CLASSES)
        _model.to(device)

        _counts = np.bincount([cid for _, cid in _split_rows['train']], minlength=NUM_CLASSES).astype(np.float64)
        _cw = _counts.sum() / np.maximum(_counts * NUM_CLASSES, 1.0)
        _criterion = nn.CrossEntropyLoss(weight=torch.from_numpy(_cw).float().to(device))

        # Simple abbreviated training for multi-seed (HEAD_EPOCHS + fine-tune)
        for param in _model.features.parameters():
            param.requires_grad = False
        _opt = torch.optim.AdamW((p for p in _model.parameters() if p.requires_grad), lr=1e-3, weight_decay=1e-4)
        _scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

        _best_val = -1.0
        _best_state = None
        for ep in range(MAX_EPOCHS):
            if ep == HEAD_EPOCHS:
                for param in _model.features.parameters():
                    param.requires_grad = True
                _opt = torch.optim.AdamW([
                    {'params': _model.features.parameters(), 'lr': 5e-5},
                    {'params': _model.classifier.parameters(), 'lr': 5e-4},
                ], weight_decay=1e-4)

            _model.train()
            for imgs, labs in _loaders['train']:
                imgs, labs = imgs.to(device), labs.to(device)
                _opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type=='cuda'):
                    out = _model(imgs)
                    loss = _criterion(out, labs)
                _scaler.scale(loss).backward()
                _scaler.step(_opt)
                _scaler.update()

            _model.eval()
            _c = _t = 0
            with torch.no_grad():
                for imgs, labs in _loaders['val']:
                    imgs, labs = imgs.to(device), labs.to(device)
                    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type=='cuda'):
                        _c += (_model(imgs).argmax(1) == labs).sum().item()
                        _t += labs.numel()
            _vacc = _c / max(_t, 1)
            if _vacc > _best_val:
                _best_val = _vacc
                _best_state = deepcopy(_model.state_dict())
            print(f'  Epoch {ep+1}/{MAX_EPOCHS} — val_acc={_vacc:.4f} (best={_best_val:.4f})')

        # Test with best model
        _model.load_state_dict(_best_state)
        _model.eval()
        _yt = []; _yp = []
        with torch.no_grad():
            for imgs, labs in _loaders['test']:
                imgs = imgs.to(device)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type=='cuda'):
                    preds = _model(imgs).argmax(1).cpu().numpy()
                _yp.append(preds)
                _yt.append(labs.numpy())
        _yt = np.concatenate(_yt); _yp = np.concatenate(_yp)
        _acc = (_yt == _yp).mean()
        _f1 = f1_score(_yt, _yp, labels=np.arange(NUM_CLASSES), average='macro', zero_division=0)
        multi_results.append({'seed': test_seed, 'test_accuracy': _acc, 'macro_f1': _f1})
        print(f'  → Test accuracy: {_acc:.4f}, Macro-F1: {_f1:.4f}')

    # Summary
    accs = [r['test_accuracy'] for r in multi_results]
    f1s  = [r['macro_f1'] for r in multi_results]
    print(f'\n{"="*50}')
    print(f'Multi-seed summary ({len(MULTI_SEEDS)} runs):')
    print(f'  Test Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}')
    print(f'  Macro-F1:      {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
    print(f'{"="*50}')

    (RUN_ROOT / 'multi_seed_results.json').write_text(json.dumps({
        'seeds': MULTI_SEEDS,
        'results': multi_results,
        'mean_accuracy': float(np.mean(accs)),
        'std_accuracy': float(np.std(accs)),
        'mean_f1': float(np.mean(f1s)),
        'std_f1': float(np.std(f1s)),
    }, indent=2) + '\n')
    print('Saved to', RUN_ROOT / 'multi_seed_results.json')
else:
    print('Multi-seed disabled. Set RUN_MULTI_SEED=True above to run.')


## 9. Export du modèle

In [ ]:
import zipfile

model.eval().cpu()
example = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
with torch.no_grad():
    traced = torch.jit.trace(model, example)
traced.save(str(RUN_ROOT / 'plantdoc_classifier.ts'))

(RUN_ROOT / 'model_config.json').write_text(json.dumps({
    'task': 'crop_disease_classification',
    'backbone': BACKBONE,
    'pretrained_on': 'imagenet',
    'input_size': [IMAGE_SIZE, IMAGE_SIZE],
    'normalization_mean': [0.485, 0.456, 0.406],
    'normalization_std': [0.229, 0.224, 0.225],
    'class_names': CLASS_NAMES,
    'dataset': DATASET_ID,
    'license': 'CC BY 4.0 (PlantDoc)',
}, indent=2) + '\n')

torch.save(model.state_dict(), RUN_ROOT / 'plantdoc_classifier_state_dict.pt')

try:
    torch.onnx.export(model, example, str(RUN_ROOT / 'plantdoc_classifier.onnx'),
                      input_names=['image'], output_names=['logits'],
                      dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
                      opset_version=17, do_constant_folding=True)
except Exception as e:
    print('ONNX export skipped:', e)

package = WORK_ROOT / f'plantdoc_{BACKBONE}_imagenet_package.zip'
with zipfile.ZipFile(package, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ('plantdoc_classifier.ts', 'plantdoc_classifier_state_dict.pt',
                 'plantdoc_classifier.onnx', 'model_config.json', 'best.pt',
                 'latest.pt', 'history.csv', 'test_metrics.json', 'confusion_matrix.png'):
        path = RUN_ROOT / name
        if path.exists():
            archive.write(path, name)
print('✅ Export package:', package)


## 10. Exemples de prédictions

In [ ]:
model.to(device)
model.eval()

correct_idx = [i for i in range(len(y_true)) if y_true[i] == y_pred[i]]
wrong_idx = [i for i in range(len(y_true)) if y_true[i] != y_pred[i]]

print(f'✅ Correctes: {len(correct_idx)} ({len(correct_idx)/len(y_true)*100:.1f}%)')
print(f'❌ Incorrectes: {len(wrong_idx)} ({len(wrong_idx)/len(y_true)*100:.1f}%)')

def show_preds(indices, title, n=8):
    if not indices:
        print(f'{title}: aucune')
        return
    sample = random.sample(indices, min(n, len(indices)))
    ncols = min(4, len(sample))
    nrows = (len(sample) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    axes = axes.flat
    fig.suptitle(title, fontsize=14)
    for ax, i in zip(axes, sample):
        test_row = split_rows['test'][i]
        cache_key = test_row[0]
        img = image_cache[cache_key]
        ax.imshow(img)
        true_name = CLASS_NAMES[y_true[i]].split('___')[-1]
        pred_name = CLASS_NAMES[y_pred[i]].split('___')[-1]
        conf = y_score[i][y_pred[i]] * 100
        color = 'green' if y_true[i] == y_pred[i] else 'red'
        ax.set_title(f'Vrai: {true_name}\nPrédit: {pred_name} ({conf:.0f}%)',
                     fontsize=8, color=color)
        ax.axis('off')
    for ax in list(axes)[len(sample):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_preds(correct_idx, '✅ Prédictions correctes')
show_preds(wrong_idx, '❌ Prédictions incorrectes')


## 11. Conclusion

### Ce notebook a fait :
1. ✅ Chargé PlantDoc (images terrain) depuis Hugging Face
2. ✅ Mappé et nettoyé les données (14 classes de pomme, maïs, tomate)
3. ✅ Vérifié l'absence de fuite de données entre train et test
4. ✅ Pré-redimensionné les images en RAM pour accélérer l'entraînement
5. ✅ Chargé ConvNeXt-Tiny avec les poids **ImageNet** (sans PlantVillage)
6. ✅ Entraîné (tête seule → fine-tuning complet)
7. ✅ Testé avec accuracy, macro-F1, confusion matrix, précision par plante
8. ✅ Exporté le modèle (TorchScript, state_dict, ONNX, package zip)

### Pour le rapport
- Comparer ConvNeXt-Tiny (ImageNet → PlantDoc) vs ConvNeXt-Tiny (ImageNet → PlantVillage)
- Analyser les classes les plus difficiles (confusion matrix)
- Discuter l'impact du type d'images (labo vs terrain)
- Utiliser les résultats multi-seed pour la robustesse (si exécuté)
